# Pipeline Silver : Kafka → Delta Lake (Données Nettoyées)

## Objectif
Implémenter la deuxième étape de l'architecture Médaillon : **Silver** (données nettoyées)

1. Consommer des messages depuis un topic Kafka
2. Appliquer des **transformations de qualité** (nettoyage approfondi, validation, normalisation)
3. Filtrer les données invalides
4. Enrichir avec des colonnes calculées métier
5. Écrire les données nettoyées dans Delta Lake (niveau Silver)

## Architecture Médaillon

```
Kafka Topic (sensor-data-iot)
    ↓
SILVER (ce notebook) : Nettoyage & Validation
    ↓
GOLD (notebook 03) : Agrégations & Analytics
```

## Principe Silver

**Silver = Données nettoyées** : Qualité garantie pour l'analyse
- ✅ Nettoyage approfondi (valeurs nulles, formats)
- ✅ Validation des plages de valeurs
- ✅ Normalisation des formats
- ✅ Filtrage des données invalides
- ✅ Enrichissement avec colonnes calculées
- ✅ Score de qualité des données
- ❌ Pas d'agrégations → Gold

## Contexte SmartTech
Cette pipeline consomme les données IoT depuis Kafka (message broker) pour :
- **Débit élevé** : Traiter des milliers de messages par seconde
- **Tolérance aux pannes** : Gestion automatique des offsets via checkpoints
- **Scalabilité** : Parallélisme via partitions Kafka
- **Temps réel** : Latence faible pour détection rapide d'anomalies

## Concepts Kafka importants

### Offsets
Les **offsets** sont des identifiants uniques pour chaque message dans une partition Kafka. Ils permettent à Spark de suivre la position de lecture et de reprendre après une panne.

### Partitions
Les **partitions** permettent de distribuer les données et le traitement. Chaque partition peut être traitée en parallèle, améliorant les performances.

### Consumer Groups
Les **consumer groups** permettent à plusieurs consommateurs de travailler ensemble pour consommer un topic. Spark utilise un consumer group pour gérer la consommation distribuée.

## 1. Configuration de l'environnement Spark

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *
import os
import json

# Configuration des chemins depuis les variables d'environnement (avec valeurs par défaut)
DELTA_SILVER_PATH = os.getenv("DELTA_SILVER_PATH", "/opt/spark/delta/silver")
CHECKPOINT_SILVER_PATH = os.getenv("CHECKPOINT_SILVER_PATH", "/opt/spark/checkpoints/silver")

# Configuration Kafka depuis les variables d'environnement
# Détection automatique : utiliser kafka:29092 dans Docker, localhost:9092 en local
# Vérifier si on est dans un conteneur Docker (meilleure détection)
is_docker = os.path.exists("/opt/spark") or os.path.exists("/.dockerenv")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", 
    "kafka:29092" if is_docker else "localhost:9092")
KAFKA_TOPIC_IOT = os.getenv("KAFKA_TOPIC_IOT", "sensor-data-iot")
KAFKA_CONSUMER_GROUP = os.getenv("KAFKA_CONSUMER_GROUP", "spark-streaming-consumer")

# Afficher la configuration détectée
print(f"🔍 Détection environnement : {'Docker' if is_docker else 'Local'}")
print(f"   Kafka Bootstrap Servers : {KAFKA_BOOTSTRAP_SERVERS}")

# Configuration Spark depuis les variables d'environnement
SPARK_APP_NAME = os.getenv("SPARK_APP_NAME", "SmartTech-Silver-Pipeline")

# Créer la session Spark avec support Delta Lake et Kafka
# IMPORTANT : Le package spark-sql-kafka doit être chargé explicitement
# Format : org.apache.spark:spark-sql-kafka-0-10_2.12:VERSION_SPARK
# Si les JARs sont pré-téléchargés dans /opt/spark/jars/, on les charge directement
# Sinon, on utilise spark.jars.packages pour télécharger depuis Maven
kafka_jars_path = "/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.4.0.jar"
kafka_clients_path = "/opt/spark/jars/kafka-clients-3.3.2.jar"

builder = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Charger les JARs Kafka : préférence pour les JARs pré-téléchargés, sinon télécharger depuis Maven
if os.path.exists(kafka_jars_path) and os.path.exists(kafka_clients_path):
    # Utiliser les JARs pré-téléchargés dans /opt/spark/jars/
    builder = builder.config("spark.jars", f"{kafka_jars_path},{kafka_clients_path}")
    print("💡 Utilisation des JARs Kafka pré-téléchargés")
else:
    # Télécharger depuis Maven Central
    builder = builder.config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0")
    print("💡 Téléchargement des JARs Kafka depuis Maven Central")

builder = builder.master(os.getenv("SPARK_MASTER", "local[*]"))

# Utiliser configure_spark_with_delta_pip() qui utilise les JARs installés par pip (delta-spark==2.4.0)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Note importante : Le package spark-sql-kafka est téléchargé automatiquement lors de la première utilisation
# Cela peut prendre quelques secondes lors du premier appel à readStream.format("kafka")
# Si vous voyez une erreur "Failed to find data source: kafka", attendez quelques secondes et réessayez
#
# ⚠️  IMPORTANT : Si vous rencontrez l'erreur NoClassDefFoundError: KafkaConfigUpdater,
# redémarrez le kernel Jupyter (Kernel → Restart) et réexécutez cette cellule pour
# recharger la session Spark avec la nouvelle configuration.

print("✓ Spark Session créée avec succès")
print(f"✓ Version Spark : {spark.version}")
print(f"✓ DELTA_SILVER_PATH : {DELTA_SILVER_PATH}")
print(f"✓ CHECKPOINT_SILVER_PATH : {CHECKPOINT_SILVER_PATH}")
print(f"✓ Kafka Bootstrap Servers : {KAFKA_BOOTSTRAP_SERVERS}")
print(f"✓ Kafka Topic : {KAFKA_TOPIC_IOT}")
print(f"✓ Consumer Group : {KAFKA_CONSUMER_GROUP}")

🔍 Détection environnement : Docker
   Kafka Bootstrap Servers : kafka:29092
💡 Utilisation des JARs Kafka pré-téléchargés
:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-023a885d-4cf8-44f3-8063-c5db8ab586c1;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (318ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (55ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (86ms)
:: resolution report :: resolve 1705ms :: artifacts dl 462ms
	:

✓ Spark Session créée avec succès
✓ Version Spark : 3.4.0
✓ DELTA_SILVER_PATH : /opt/spark/delta/silver
✓ CHECKPOINT_SILVER_PATH : /opt/spark/checkpoints/silver
✓ Kafka Bootstrap Servers : kafka:29092
✓ Kafka Topic : sensor-data-iot
✓ Consumer Group : spark-streaming-consumer


In [2]:
# Nettoyage des données Delta Lake Silver et checkpoints (optionnel)
import shutil
import os
from pathlib import Path

# Définir les chemins si pas encore définis
if 'DELTA_SILVER_PATH' not in globals():
    DELTA_SILVER_PATH = os.getenv("DELTA_SILVER_PATH", "/opt/spark/delta/silver")
if 'CHECKPOINT_SILVER_PATH' not in globals():
    CHECKPOINT_SILVER_PATH = os.getenv("CHECKPOINT_SILVER_PATH", "/opt/spark/checkpoints/silver")

print("🧹 Nettoyage des données Delta Lake Silver et checkpoints...")
print("⚠️  ATTENTION : Cette opération supprime toutes les données existantes !")
print(f"   - Delta Silver : {DELTA_SILVER_PATH}")
print(f"   - Checkpoints : {CHECKPOINT_SILVER_PATH}")

# Nettoyage actif (mettez True pour nettoyer)
CLEAN_SILVER = False  # ⚠️ Changez à True pour nettoyer

if CLEAN_SILVER:
    try:
        # Arrêter toutes les queries en cours si elles existent
        try:
            if 'query_silver' in globals():
                query_silver.stop()
                print("✓ Query Silver arrêtée")
        except:
            pass
        
        # Supprimer la table Delta Silver
        silver_path = Path(DELTA_SILVER_PATH)
        if silver_path.exists():
            shutil.rmtree(str(silver_path))
            print(f"✓ Table Delta Silver supprimée : {DELTA_SILVER_PATH}")
        else:
            print(f"ℹ️  Table Delta Silver n'existe pas encore : {DELTA_SILVER_PATH}")
        
        # Supprimer les checkpoints
        checkpoint_path = Path(CHECKPOINT_SILVER_PATH)
        if checkpoint_path.exists():
            shutil.rmtree(str(checkpoint_path))
            print(f"✓ Checkpoints supprimés : {CHECKPOINT_SILVER_PATH}")
        else:
            print(f"ℹ️  Checkpoints n'existent pas encore : {CHECKPOINT_SILVER_PATH}")
        
        print("\n✅ Nettoyage terminé - Vous pouvez repartir de zéro")
        print("💡 N'oubliez pas de remettre CLEAN_SILVER = False après le nettoyage")
    except Exception as e:
        print(f"❌ Erreur lors du nettoyage : {e}")
else:
    print("\n💡 Pour effectuer le nettoyage, mettez CLEAN_SILVER = True et réexécutez cette cellule")

🧹 Nettoyage des données Delta Lake Silver et checkpoints...
⚠️  ATTENTION : Cette opération supprime toutes les données existantes !
   - Delta Silver : /opt/spark/delta/silver
   - Checkpoints : /opt/spark/checkpoints/silver

💡 Pour effectuer le nettoyage, mettez CLEAN_SILVER = True et réexécutez cette cellule


## 2. Lancement du producteur Kafka

**Important** : Le producteur génère des données **BRUTES** (comme dans Bronze) avec :
- Valeurs null ou manquantes
- Formats de timestamp variés
- Valeurs hors limites
- Types incorrects

C'est à la pipeline Silver de **nettoyer** ces données brutes.

In [3]:
# Lancement automatique du producteur Kafka (si nécessaire)
# Ce producteur génère des données BRUTES (comme dans Bronze) et les envoie dans le topic Kafka
# IMPORTANT : Le producteur continue à tourner même si la pipeline s'arrête
# Utilisez la cellule "Arrêt du producteur Kafka" pour l'arrêter proprement
import subprocess
import os
import time
import signal
from pathlib import Path

# Configuration
SCRIPT_DIR = Path("/opt/spark/scripts")
PRODUCER_SCRIPT = SCRIPT_DIR / "start_kafka_producer.sh"
KAFKA_PRODUCER_PY = SCRIPT_DIR / "kafka_sensor_producer.py"

# Variable globale pour stocker le processus du producteur (pour pouvoir l'arrêter)
# Initialiser à None si pas encore défini
if 'kafka_producer_process' not in globals():
    kafka_producer_process = None

# Option : mettre AUTO_START_PRODUCER = False pour lancer manuellement
AUTO_START_PRODUCER = os.getenv("AUTO_START_PRODUCER", "true").lower() == "true"

print("🔍 Vérification du producteur Kafka...")
print(f"   Environnement détecté : {'Docker' if is_docker else 'Local'}")
print(f"   Kafka Bootstrap Servers : {KAFKA_BOOTSTRAP_SERVERS}")
print(f"   Kafka Topic : {KAFKA_TOPIC_IOT}")
print("💡 Le producteur génère des données BRUTES (à nettoyer dans Silver)")

if AUTO_START_PRODUCER:
    # Vérifier si le script existe
    if not PRODUCER_SCRIPT.exists() and not KAFKA_PRODUCER_PY.exists():
        print("⚠️  Scripts de producteur Kafka non trouvés")
        print(f"   Cherché dans : {SCRIPT_DIR}")
        print("💡 Le producteur doit être lancé manuellement")
    else:
        # Vérifier si le producteur est déjà en cours d'exécution
        # (vérifier les processus Python qui exécutent kafka_sensor_producer.py)
        try:
            result = subprocess.run(
                ["pgrep", "-f", "kafka_sensor_producer.py"],
                capture_output=True,
                text=True
            )
            if result.returncode == 0:
                pids = result.stdout.strip().split('\n')
                print(f"✓ Producteur Kafka déjà en cours d'exécution (PID: {', '.join(pids)})")
                print("💡 Pour l'arrêter, utilisez la cellule 'Arrêt du producteur Kafka'")
            else:
                print("🚀 Lancement du producteur Kafka en arrière-plan...")
                print(f"   Script : {PRODUCER_SCRIPT}")
                
                # Lancer le script en arrière-plan
                if PRODUCER_SCRIPT.exists():
                    process = subprocess.Popen(
                        ["bash", str(PRODUCER_SCRIPT)],
                        stdout=subprocess.PIPE,
                        stderr=subprocess.PIPE,
                        cwd=str(SCRIPT_DIR)
                    )
                    kafka_producer_process = process
                    print(f"✓ Producteur Kafka lancé (PID: {process.pid})")
                    print("   ⏳ Attente de 5 secondes pour que Kafka soit prêt...")
                    time.sleep(5)
                elif KAFKA_PRODUCER_PY.exists():
                    # Lancer directement le script Python
                    env = os.environ.copy()
                    env["KAFKA_BOOTSTRAP_SERVERS"] = KAFKA_BOOTSTRAP_SERVERS
                    env["KAFKA_TOPIC_IOT"] = KAFKA_TOPIC_IOT
                    env["DATA_DIR"] = os.getenv("DATA_DIR", "/opt/spark/data")
                    env["CONTINUOUS_MODE"] = "true"
                    
                    process = subprocess.Popen(
                        ["python3", str(KAFKA_PRODUCER_PY)],
                        stdout=subprocess.PIPE,
                        stderr=subprocess.PIPE,
                        cwd=str(SCRIPT_DIR),
                        env=env
                    )
                    kafka_producer_process = process
                    print(f"✓ Producteur Kafka lancé (PID: {process.pid})")
                    print("   ⏳ Attente de 5 secondes pour que Kafka soit prêt...")
                    time.sleep(5)
        except FileNotFoundError:
            # pgrep n'est pas disponible, essayer quand même de lancer
            print("🚀 Lancement du producteur Kafka...")
            if PRODUCER_SCRIPT.exists():
                process = subprocess.Popen(
                    ["bash", str(PRODUCER_SCRIPT)],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL,
                    cwd=str(SCRIPT_DIR)
                )
                kafka_producer_process = process
                print(f"✓ Producteur Kafka lancé en arrière-plan (PID: {process.pid})")
                print("   ⏳ Attente de 5 secondes pour que Kafka soit prêt...")
                time.sleep(5)
            else:
                print("⚠️  Script de producteur non trouvé")
                print("💡 Lancez manuellement : python3 /opt/spark/scripts/kafka_sensor_producer.py")
else:
    print("ℹ️  Lancement automatique désactivé (AUTO_START_PRODUCER=False)")
    print("💡 Pour lancer le producteur manuellement :")
    print("   python3 /opt/spark/scripts/kafka_sensor_producer.py")
    print("   ou")
    print("   bash /opt/spark/scripts/start_kafka_producer.sh")

🔍 Vérification du producteur Kafka...
   Environnement détecté : Docker
   Kafka Bootstrap Servers : kafka:29092
   Kafka Topic : sensor-data-iot
💡 Le producteur génère des données BRUTES (à nettoyer dans Silver)
🚀 Lancement du producteur Kafka en arrière-plan...
   Script : /opt/spark/scripts/start_kafka_producer.sh
✓ Producteur Kafka lancé (PID: 251)
   ⏳ Attente de 5 secondes pour que Kafka soit prêt...


In [4]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *
import os
import json

# Configuration des chemins depuis les variables d'environnement (avec valeurs par défaut)
DELTA_SILVER_PATH = os.getenv("DELTA_SILVER_PATH", "/opt/spark/delta/silver")
CHECKPOINT_SILVER_PATH = os.getenv("CHECKPOINT_SILVER_PATH", "/opt/spark/checkpoints/silver")

# Configuration Kafka depuis les variables d'environnement
# Détection automatique : utiliser kafka:29092 dans Docker, localhost:9092 en local
# Vérifier si on est dans un conteneur Docker (meilleure détection)
is_docker = os.path.exists("/opt/spark") or os.path.exists("/.dockerenv")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", 
    "kafka:29092" if is_docker else "localhost:9092")
KAFKA_TOPIC_IOT = os.getenv("KAFKA_TOPIC_IOT", "sensor-data-iot")
KAFKA_CONSUMER_GROUP = os.getenv("KAFKA_CONSUMER_GROUP", "spark-streaming-consumer")

# Afficher la configuration détectée
print(f"🔍 Détection environnement : {'Docker' if is_docker else 'Local'}")
print(f"   Kafka Bootstrap Servers : {KAFKA_BOOTSTRAP_SERVERS}")

# Configuration Spark depuis les variables d'environnement
SPARK_APP_NAME = os.getenv("SPARK_APP_NAME", "SmartTech-Silver-Pipeline")

# Créer la session Spark avec support Delta Lake et Kafka
# IMPORTANT : Le package spark-sql-kafka doit être chargé explicitement
# Format : org.apache.spark:spark-sql-kafka-0-10_2.12:VERSION_SPARK
# Si les JARs sont pré-téléchargés dans /opt/spark/jars/, on les charge directement
# Sinon, on utilise spark.jars.packages pour télécharger depuis Maven
kafka_jars_path = "/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.4.0.jar"
kafka_clients_path = "/opt/spark/jars/kafka-clients-3.3.2.jar"

builder = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Charger les JARs Kafka : préférence pour les JARs pré-téléchargés, sinon télécharger depuis Maven
if os.path.exists(kafka_jars_path) and os.path.exists(kafka_clients_path):
    # Utiliser les JARs pré-téléchargés dans /opt/spark/jars/
    builder = builder.config("spark.jars", f"{kafka_jars_path},{kafka_clients_path}")
    print("💡 Utilisation des JARs Kafka pré-téléchargés")
else:
    # Télécharger depuis Maven Central
    builder = builder.config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0")
    print("💡 Téléchargement des JARs Kafka depuis Maven Central")

builder = builder.master(os.getenv("SPARK_MASTER", "local[*]"))

# Utiliser configure_spark_with_delta_pip() qui utilise les JARs installés par pip (delta-spark==2.4.0)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Note importante : Le package spark-sql-kafka est téléchargé automatiquement lors de la première utilisation
# Cela peut prendre quelques secondes lors du premier appel à readStream.format("kafka")
# Si vous voyez une erreur "Failed to find data source: kafka", attendez quelques secondes et réessayez
#
# ⚠️  IMPORTANT : Si vous rencontrez l'erreur NoClassDefFoundError: KafkaConfigUpdater,
# redémarrez le kernel Jupyter (Kernel → Restart) et réexécutez cette cellule pour
# recharger la session Spark avec la nouvelle configuration.

print("✓ Spark Session créée avec succès")
print(f"✓ Version Spark : {spark.version}")
print(f"✓ DELTA_SILVER_PATH : {DELTA_SILVER_PATH}")
print(f"✓ CHECKPOINT_SILVER_PATH : {CHECKPOINT_SILVER_PATH}")
print(f"✓ Kafka Bootstrap Servers : {KAFKA_BOOTSTRAP_SERVERS}")
print(f"✓ Kafka Topic : {KAFKA_TOPIC_IOT}")
print(f"✓ Consumer Group : {KAFKA_CONSUMER_GROUP}")

🔍 Détection environnement : Docker
   Kafka Bootstrap Servers : kafka:29092
💡 Utilisation des JARs Kafka pré-téléchargés
✓ Spark Session créée avec succès
✓ Version Spark : 3.4.0
✓ DELTA_SILVER_PATH : /opt/spark/delta/silver
✓ CHECKPOINT_SILVER_PATH : /opt/spark/checkpoints/silver
✓ Kafka Bootstrap Servers : kafka:29092
✓ Kafka Topic : sensor-data-iot
✓ Consumer Group : spark-streaming-consumer


## 3. Vérification de la connectivité Kafka

In [5]:
# Vérifier que Kafka est accessible et que le topic existe
print("🔍 Vérification de la connectivité Kafka...")
print(f"   Broker : {KAFKA_BOOTSTRAP_SERVERS}")
print(f"   Topic : {KAFKA_TOPIC_IOT}")

# Forcer le chargement du package Kafka avec retry automatique
import time
max_retries = 10  # Augmenté pour laisser plus de temps au téléchargement
base_delay = 10  # Délai de base en secondes (augmenté)

print("\n⏳ Vérification du package spark-sql-kafka...")
print("   (Le téléchargement peut prendre 30-90 secondes lors de la première utilisation)")
print("   Le package est téléchargé depuis Maven Central...")

for attempt in range(max_retries):
    try:
        # Tenter de créer un DataFrame Kafka pour forcer le chargement du package
        test_df = spark.readStream \
            .format("kafka") \
            .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
            .option("subscribe", KAFKA_TOPIC_IOT) \
            .option("startingOffsets", "earliest") \
            .option("maxOffsetsPerTrigger", 1) \
            .load()
        
        print(f"\n✓ Package spark-sql-kafka chargé avec succès (tentative {attempt + 1}/{max_retries})")
        print("✓ Connexion à Kafka réussie")
        print("✓ Topic accessible")
        print(f"\n📊 Schéma des données Kafka :")
        test_df.printSchema()
        break
        
    except Exception as e:
        error_msg = str(e)
        if "Failed to find data source: kafka" in error_msg:
            if attempt < max_retries - 1:
                # Délai progressif : augmente avec chaque tentative
                current_delay = base_delay + (attempt * 3)
                # Utiliser builtins.sum() pour éviter le conflit avec pyspark.sql.functions.sum
                import builtins
                elapsed_time = builtins.sum(base_delay + (i * 3) for i in range(attempt + 1))
                print(f"   ⏳ Tentative {attempt + 1}/{max_retries} : Package en cours de téléchargement...")
                print(f"   ⏳ Temps écoulé : ~{elapsed_time} secondes")
                print(f"   ⏳ Attente de {current_delay} secondes avant de réessayer...")
                time.sleep(current_delay)
            else:
                # Dernière tentative échouée
                import builtins
                total_time = builtins.sum(base_delay + (i * 3) for i in range(max_retries - 1))
                print(f"\n❌ Erreur : Package spark-sql-kafka non chargé après {max_retries} tentatives (~{total_time} secondes)")
                print(f"\n💡 Solutions :")
                print(f"   1. ⏳ Attendez encore 30-60 secondes et réexécutez cette cellule")
                print(f"   2. Redémarrez le kernel Jupyter (Kernel → Restart) et réessayez")
                print(f"   3. Vérifiez votre connexion Internet depuis le conteneur")
                print(f"   4. Reconstruisez l'image Docker pour pré-télécharger le package")
                print(f"\n   Le package sera téléchargé automatiquement depuis Maven Central")
                print(f"   Format : org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0")
                print(f"   Taille : ~2-3 MB (peut prendre du temps selon la connexion)")
        else:
            # Autre erreur (connexion Kafka, topic, etc.)
            print(f"❌ Erreur de connexion à Kafka : {e}")
            print(f"\n💡 Vérifiez que :")
            print(f"   1. Kafka est démarré (docker-compose up -d)")
            print(f"   2. Le topic '{KAFKA_TOPIC_IOT}' existe")
            print(f"   3. Le broker est accessible sur {KAFKA_BOOTSTRAP_SERVERS}")
            print(f"\n   Pour créer le topic manuellement :")
            print(f"   docker exec -it kafka kafka-topics --create --topic {KAFKA_TOPIC_IOT} --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1")
            break

🔍 Vérification de la connectivité Kafka...
   Broker : kafka:29092
   Topic : sensor-data-iot

⏳ Vérification du package spark-sql-kafka...
   (Le téléchargement peut prendre 30-90 secondes lors de la première utilisation)
   Le package est téléchargé depuis Maven Central...

✓ Package spark-sql-kafka chargé avec succès (tentative 1/10)
✓ Connexion à Kafka réussie
✓ Topic accessible

📊 Schéma des données Kafka :
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## 4. Définition du schéma des données

In [6]:
# Schéma attendu pour les données IoT (niveau Silver)
# Ce schéma est plus strict que Bronze car les données sont déjà nettoyées
sensor_schema = StructType([
    StructField("sensor_id", StringType(), False),  # Non nullable en Silver
    StructField("timestamp", StringType(), False),  # Non nullable
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("energy_consumption", DoubleType(), True),
    StructField("anomaly_detected", BooleanType(), False),  # Non nullable
    StructField("building_id", StringType(), False),  # Non nullable
    StructField("sensor_type", StringType(), False),  # Non nullable
    StructField("location", StringType(), True)
])

print("✓ Schéma des données défini pour le niveau Silver")
print("\n📋 Schéma :")
# Afficher le schéma en créant un DataFrame vide avec ce schéma
empty_df = spark.createDataFrame([], sensor_schema)
empty_df.printSchema()

✓ Schéma des données défini pour le niveau Silver

📋 Schéma :
root
 |-- sensor_id: string (nullable = false)
 |-- timestamp: string (nullable = false)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- energy_consumption: double (nullable = true)
 |-- anomaly_detected: boolean (nullable = false)
 |-- building_id: string (nullable = false)
 |-- sensor_type: string (nullable = false)
 |-- location: string (nullable = true)



## 5. Lecture du flux Kafka

In [7]:
# Configuration de la source Kafka
# IMPORTANT : Les données Kafka sont au format binaire
# La colonne 'value' contient les messages JSON en bytes
# earliest : lire depuis le début du topic (pour les tests)
# latest : lire uniquement les nouveaux messages (pour la production)
# NOTE : Ne pas utiliser kafka.group.id explicitement - Spark gère automatiquement les consumer groups
# via les checkpoints. Chaque query streaming a son propre consumer group basé sur le checkpoint.
# failOnDataLoss=false : ne pas échouer si des données sont perdues (utile pour les tests)
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC_IOT) \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

print("✓ Source Kafka configurée")
print(f"\n📊 Schéma des données Kafka (avant parsing) :")
kafka_stream.printSchema()

print("\n💡 Colonnes disponibles :")
print("   - key : Clé du message (sensor_id)")
print("   - value : Valeur du message (JSON en bytes)")
print("   - topic : Nom du topic")
print("   - partition : Partition Kafka")
print("   - offset : Offset du message")
print("   - timestamp : Timestamp Kafka")

✓ Source Kafka configurée

📊 Schéma des données Kafka (avant parsing) :
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)


💡 Colonnes disponibles :
   - key : Clé du message (sensor_id)
   - value : Valeur du message (JSON en bytes)
   - topic : Nom du topic
   - partition : Partition Kafka
   - offset : Offset du message
   - timestamp : Timestamp Kafka


In [8]:
# 🔧 CORRECTION : Arrêter et recréer les streams avec la nouvelle configuration
# Cette cellule résout le problème de NoClassDefFoundError: KafkaConfigUpdater
# en recréant tous les streams sans l'option kafka.group.id

print("🔧 Correction de la configuration Kafka...")
print("   Arrêt des queries existantes...")

# Arrêter toutes les queries en cours
try:
    if 'query_silver' in globals() and query_silver is not None:
        query_silver.stop()
        print("✓ query_silver arrêtée")
except Exception as e:
    print(f"⚠️  Erreur lors de l'arrêt de query_silver : {e}")

try:
    if 'query_silver_restart' in globals() and query_silver_restart is not None:
        query_silver_restart.stop()
        print("✓ query_silver_restart arrêtée")
except Exception as e:
    print(f"⚠️  Erreur lors de l'arrêt de query_silver_restart : {e}")

# Vérifier que les JARs Kafka sont présents
print("\n🔍 Vérification des JARs Kafka...")
kafka_jars_path = "/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.4.0.jar"
kafka_clients_path = "/opt/spark/jars/kafka-clients-3.3.2.jar"

if os.path.exists(kafka_jars_path):
    print(f"✓ JAR Kafka trouvé : {kafka_jars_path}")
else:
    print(f"⚠️  JAR Kafka non trouvé : {kafka_jars_path}")

if os.path.exists(kafka_clients_path):
    print(f"✓ JAR kafka-clients trouvé : {kafka_clients_path}")
else:
    print(f"⚠️  JAR kafka-clients non trouvé : {kafka_clients_path}")

# Recréer le stream Kafka SANS kafka.group.id
print("\n🔄 Recréation du stream Kafka avec la nouvelle configuration...")
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC_IOT) \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

print("✓ Stream Kafka recréé (sans kafka.group.id)")

# Recréer toute la chaîne de transformations
print("🔄 Recréation de la chaîne de transformations...")

# Parsing JSON
parsed_stream = kafka_stream \
    .select(
        col("key").cast("string").alias("kafka_key"),
        col("value").cast("string").alias("json_string"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    ) \
    .filter(col("json_string").isNotNull()) \
    .withColumn("data", from_json(col("json_string"), sensor_schema)) \
    .select("data.*", "kafka_key", "topic", "partition", "offset", "kafka_timestamp")

# Normalisation et nettoyage
normalized_stream = parsed_stream \
    .withColumn("timestamp_normalized",
        when(
            col("timestamp").rlike(".*Z$"),
            regexp_replace(col("timestamp"), "Z$", "+00:00")
        ).otherwise(
            concat(col("timestamp"), lit("+00:00"))
        )
    ) \
    .withColumn("timestamp",
        to_timestamp(
            col("timestamp_normalized"),
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        )
    ) \
    .drop("timestamp_normalized")

cleaned_stream = normalized_stream \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_quality_score",
        when(
            (col("sensor_id").isNotNull()) &
            (col("timestamp").isNotNull()) &
            (col("building_id").isNotNull()) &
            (col("sensor_type").isNotNull()),
            lit(100)
        ).otherwise(lit(50))
    ) \
    .withColumn("temperature_status",
        when(col("temperature").isNull(), lit("missing"))
        .when((col("temperature") < -50) | (col("temperature") > 60), lit("out_of_range"))
        .otherwise(lit("valid"))
    ) \
    .withColumn("humidity_status",
        when(col("humidity").isNull(), lit("missing"))
        .when((col("humidity") < 0) | (col("humidity") > 100), lit("out_of_range"))
        .otherwise(lit("valid"))
    ) \
    .withColumn("energy_status",
        when(col("energy_consumption").isNull(), lit("missing"))
        .when(col("energy_consumption") < 0, lit("invalid"))
        .otherwise(lit("valid"))
    ) \
    .withColumn("temp_status",
        when(col("temperature").isNull(), lit("missing"))
        .when(col("temperature") > 30, lit("HIGH"))
        .otherwise(lit("NORMAL"))
    ) \
    .withColumn("energy_status_calculated",
        when(col("energy_consumption").isNull(), lit("missing"))
        .when(col("energy_consumption") > 800, lit("HIGH"))
        .otherwise(lit("NORMAL"))
    )

# Filtrage
filtered_stream = cleaned_stream \
    .filter(
        (col("sensor_id").isNotNull()) &
        (col("timestamp").isNotNull()) &
        (col("building_id").isNotNull()) &
        (col("sensor_type").isNotNull()) &
        (
            (
                (col("temperature").isNotNull()) &
                (col("temperature") >= -50) & (col("temperature") <= 60)
            ) |
            (
                (col("humidity").isNotNull()) &
                (col("humidity") >= 0) & (col("humidity") <= 100)
            ) |
            (
                (col("energy_consumption").isNotNull()) &
                (col("energy_consumption") >= 0)
            )
        )
    )

# Sélection finale
silver_stream = filtered_stream \
    .select(
        "sensor_id",
        "timestamp",
        "temperature",
        "humidity",
        "energy_consumption",
        "anomaly_detected",
        "building_id",
        "sensor_type",
        "location",
        "ingestion_timestamp",
        "data_quality_score",
        "temperature_status",
        "humidity_status",
        "energy_status",
        "temp_status",
        "energy_status_calculated"
    )

print("✓ Chaîne de transformations recréée")
print("\n✅ Configuration corrigée - Vous pouvez maintenant recréer query_silver")
print("💡 Exécutez la cellule 'Configuration de l'écriture vers Delta Lake Silver' pour démarrer la pipeline")

🔧 Correction de la configuration Kafka...
   Arrêt des queries existantes...

🔍 Vérification des JARs Kafka...
✓ JAR Kafka trouvé : /opt/spark/jars/spark-sql-kafka-0-10_2.12-3.4.0.jar
✓ JAR kafka-clients trouvé : /opt/spark/jars/kafka-clients-3.3.2.jar

🔄 Recréation du stream Kafka avec la nouvelle configuration...
✓ Stream Kafka recréé (sans kafka.group.id)
🔄 Recréation de la chaîne de transformations...
✓ Chaîne de transformations recréée

✅ Configuration corrigée - Vous pouvez maintenant recréer query_silver
💡 Exécutez la cellule 'Configuration de l'écriture vers Delta Lake Silver' pour démarrer la pipeline


## 5. Parsing des messages JSON

In [9]:
# Parser les messages JSON depuis la colonne 'value'
# La colonne 'value' est de type binary, il faut la convertir en string puis parser le JSON
parsed_stream = kafka_stream \
    .select(
        col("key").cast("string").alias("kafka_key"),
        col("value").cast("string").alias("json_string"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    ) \
    .filter(col("json_string").isNotNull()) \
    .withColumn("data", from_json(col("json_string"), sensor_schema)) \
    .select("data.*", "kafka_key", "topic", "partition", "offset", "kafka_timestamp")

print("✓ Parsing JSON effectué")
print(f"\n📊 Schéma après parsing :")
parsed_stream.printSchema()

✓ Parsing JSON effectué

📊 Schéma après parsing :
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- energy_consumption: double (nullable = true)
 |-- anomaly_detected: boolean (nullable = true)
 |-- building_id: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- location: string (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)



## 10. Arrêt du producteur Kafka

**Important** : Le producteur Kafka continue à produire des données même si la pipeline s'arrête. Utilisez cette cellule pour l'arrêter proprement.

In [10]:
# Arrêt propre du producteur Kafka
# Cette cellule arrête le producteur Kafka qui a été lancé automatiquement
import subprocess
import signal
import os

print("🛑 Arrêt du producteur Kafka...")

# Méthode 1 : Arrêter via le processus stocké dans kafka_producer_process
stopped = False
if 'kafka_producer_process' in globals() and kafka_producer_process is not None:
    try:
        if kafka_producer_process.poll() is None:  # Processus encore en cours
            print(f"   Arrêt du processus (PID: {kafka_producer_process.pid})...")
            kafka_producer_process.terminate()  # Envoie SIGTERM
            try:
                kafka_producer_process.wait(timeout=5)  # Attendre jusqu'à 5 secondes
                print("✓ Producteur Kafka arrêté proprement")
                stopped = True
            except subprocess.TimeoutExpired:
                print("   ⚠️  Le processus n'a pas répondu, envoi de SIGKILL...")
                kafka_producer_process.kill()  # Force l'arrêt
                kafka_producer_process.wait()
                print("✓ Producteur Kafka arrêté (forcé)")
                stopped = True
        else:
            print(f"   Le processus (PID: {kafka_producer_process.pid}) est déjà arrêté")
            stopped = True
    except Exception as e:
        print(f"   ⚠️  Erreur lors de l'arrêt du processus : {e}")

# Méthode 2 : Chercher et arrêter tous les processus kafka_sensor_producer.py
if not stopped:
    try:
        result = subprocess.run(
            ["pgrep", "-f", "kafka_sensor_producer.py"],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            pids = [int(pid) for pid in result.stdout.strip().split('\n') if pid]
            print(f"   Trouvé {len(pids)} processus producteur(s) Kafka")
            for pid in pids:
                try:
                    print(f"   Arrêt du processus PID {pid}...")
                    os.kill(pid, signal.SIGTERM)  # Envoie SIGTERM
                    # Attendre un peu pour voir si le processus s'arrête
                    import time
                    time.sleep(2)
                    # Vérifier si le processus existe encore
                    try:
                        os.kill(pid, 0)  # Vérifie si le processus existe
                        print(f"   ⚠️  Le processus {pid} n'a pas répondu, envoi de SIGKILL...")
                        os.kill(pid, signal.SIGKILL)  # Force l'arrêt
                    except ProcessLookupError:
                        pass  # Le processus s'est arrêté
                    print(f"✓ Processus {pid} arrêté")
                    stopped = True
                except ProcessLookupError:
                    print(f"   Le processus {pid} n'existe plus")
                except PermissionError:
                    print(f"   ⚠️  Permission refusée pour arrêter le processus {pid}")
                except Exception as e:
                    print(f"   ⚠️  Erreur lors de l'arrêt du processus {pid} : {e}")
        else:
            print("ℹ️  Aucun processus producteur Kafka trouvé")
    except FileNotFoundError:
        print("⚠️  Commande 'pgrep' non disponible")
        print("💡 Arrêtez manuellement le producteur avec : pkill -f kafka_sensor_producer.py")
    except Exception as e:
        print(f"⚠️  Erreur lors de la recherche des processus : {e}")

if stopped:
    print("\n✅ Producteur Kafka arrêté")
    print("💡 Pour le relancer, réexécutez la cellule 'Lancement automatique du producteur Kafka'")
else:
    print("\n⚠️  Aucun producteur Kafka n'a été trouvé ou arrêté")
    print("💡 Le producteur peut avoir été lancé manuellement ou être déjà arrêté")

🛑 Arrêt du producteur Kafka...
   Le processus (PID: 251) est déjà arrêté

✅ Producteur Kafka arrêté
💡 Pour le relancer, réexécutez la cellule 'Lancement automatique du producteur Kafka'


## 6. Transformations et normalisation (niveau Silver)

In [11]:
# Transformations avancées pour le niveau Silver
# Plus approfondies que Bronze car on a déjà des données partiellement nettoyées

# 1. Normalisation des timestamps
# Gérer les formats ISO 8601 avec/sans timezone
normalized_stream = parsed_stream \
    .withColumn("timestamp_normalized",
        when(
            col("timestamp").rlike(".*Z$"),  # Format avec 'Z' (UTC)
            regexp_replace(col("timestamp"), "Z$", "+00:00")
        ).otherwise(
            # Pas de timezone : ajouter '+00:00' (assume UTC)
            concat(col("timestamp"), lit("+00:00"))
        )
    ) \
    .withColumn("timestamp",
        to_timestamp(
            col("timestamp_normalized"),
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        )
    ) \
    .drop("timestamp_normalized")

# 2. Validation et nettoyage approfondi
# Appliquer des règles métier plus strictes qu'en Bronze
cleaned_stream = normalized_stream \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_quality_score",
        when(
            (col("sensor_id").isNotNull()) &
            (col("timestamp").isNotNull()) &
            (col("building_id").isNotNull()) &
            (col("sensor_type").isNotNull()),
            lit(100)  # Score parfait si toutes les colonnes requises sont présentes
        ).otherwise(lit(50))  # Score réduit sinon
    ) \
    .withColumn("temperature_status",
        when(col("temperature").isNull(), lit("missing"))
        .when((col("temperature") < -50) | (col("temperature") > 60), lit("out_of_range"))
        .otherwise(lit("valid"))
    ) \
    .withColumn("humidity_status",
        when(col("humidity").isNull(), lit("missing"))
        .when((col("humidity") < 0) | (col("humidity") > 100), lit("out_of_range"))
        .otherwise(lit("valid"))
    ) \
    .withColumn("energy_status",
        when(col("energy_consumption").isNull(), lit("missing"))
        .when(col("energy_consumption") < 0, lit("invalid"))
        .otherwise(lit("valid"))
        ) \
    .withColumn("temp_status",  # Colonnes calculées métier (déplacées depuis Bronze)
        when(col("temperature").isNull(), lit("missing"))
        .when(col("temperature") > 30, lit("HIGH"))
        .otherwise(lit("NORMAL"))
    ) \
    .withColumn("energy_status_calculated",
        when(col("energy_consumption").isNull(), lit("missing"))
        .when(col("energy_consumption") > 800, lit("HIGH"))
        .otherwise(lit("NORMAL"))
    )

print("✓ Normalisation et nettoyage approfondi effectués")
print("✓ Colonnes calculées métier ajoutées (temp_status, energy_status_calculated)")
print(f"\n📊 Schéma après nettoyage :")
cleaned_stream.printSchema()

✓ Normalisation et nettoyage approfondi effectués
✓ Colonnes calculées métier ajoutées (temp_status, energy_status_calculated)

📊 Schéma après nettoyage :
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- energy_consumption: double (nullable = true)
 |-- anomaly_detected: boolean (nullable = true)
 |-- building_id: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- location: string (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- data_quality_score: integer (nullable = false)
 |-- temperature_status: string (nullable = false)
 |-- humidity_status: string (nullable = false)
 |-- energy_status: string (nullabl

## 7. Filtrage des données valides

In [12]:
# Filtrage strict pour le niveau Silver
# Seules les données complètes et valides sont conservées
filtered_stream = cleaned_stream \
    .filter(
        (col("sensor_id").isNotNull()) &
        (col("timestamp").isNotNull()) &
        (col("building_id").isNotNull()) &
        (col("sensor_type").isNotNull()) &
        (
            # Au moins une mesure doit être présente et valide
            (
                (col("temperature").isNotNull()) &
                (col("temperature") >= -50) & (col("temperature") <= 60)
            ) |
            (
                (col("humidity").isNotNull()) &
                (col("humidity") >= 0) & (col("humidity") <= 100)
            ) |
            (
                (col("energy_consumption").isNotNull()) &
                (col("energy_consumption") >= 0)
            )
        )
    )

print("✓ Filtrage des données valides effectué")
print("\n💡 Critères de filtrage Silver :")
print("   - Toutes les colonnes requises doivent être présentes")
print("   - Au moins une mesure (température, humidité, énergie) doit être valide")
print("   - Les valeurs doivent être dans les plages acceptables")

✓ Filtrage des données valides effectué

💡 Critères de filtrage Silver :
   - Toutes les colonnes requises doivent être présentes
   - Au moins une mesure (température, humidité, énergie) doit être valide
   - Les valeurs doivent être dans les plages acceptables


## 8. Sélection des colonnes finales

In [13]:
# Sélectionner uniquement les colonnes nécessaires pour Silver
# On garde les métadonnées Kafka pour le traçabilité mais on peut les exclure
silver_stream = filtered_stream \
    .select(
        "sensor_id",
        "timestamp",
        "temperature",
        "humidity",
        "energy_consumption",
        "anomaly_detected",
        "building_id",
        "sensor_type",
        "location",
        "ingestion_timestamp",
        "data_quality_score",
        "temperature_status",
        "humidity_status",
        "energy_status",
        "temp_status",
        "energy_status_calculated"
        # Optionnel : garder kafka_key, partition, offset pour le traçabilité
    )

print("✓ Colonnes finales sélectionnées")
print(f"\n📊 Schéma final pour Silver :")
silver_stream.printSchema()

✓ Colonnes finales sélectionnées

📊 Schéma final pour Silver :
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- energy_consumption: double (nullable = true)
 |-- anomaly_detected: boolean (nullable = true)
 |-- building_id: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- location: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- data_quality_score: integer (nullable = false)
 |-- temperature_status: string (nullable = false)
 |-- humidity_status: string (nullable = false)
 |-- energy_status: string (nullable = false)
 |-- temp_status: string (nullable = false)
 |-- energy_status_calculated: string (nullable = false)



## 9. Configuration de l'écriture vers Delta Lake Silver

In [14]:
# Configuration de l'écriture vers Delta Lake Silver
# Mode update : permet de mettre à jour les données si nécessaire
# (mais on utilise append pour l'ingestion continue)
# IMPORTANT : Les checkpoints gèrent automatiquement les offsets Kafka
query_silver = silver_stream \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH) \
    .option("path", DELTA_SILVER_PATH) \
    .partitionBy("building_id", "sensor_type") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✓ Pipeline Silver démarrée")
print(f"✓ Écriture dans : {DELTA_SILVER_PATH}")
print(f"✓ Checkpoint dans : {CHECKPOINT_SILVER_PATH}")
print(f"✓ Trigger : toutes les 10 secondes")
print(f"\n💡 Les offsets Kafka sont gérés automatiquement via les checkpoints")

✓ Pipeline Silver démarrée
✓ Écriture dans : /opt/spark/delta/silver
✓ Checkpoint dans : /opt/spark/checkpoints/silver
✓ Trigger : toutes les 10 secondes

💡 Les offsets Kafka sont gérés automatiquement via les checkpoints


25/12/17 15:12:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 10. Monitoring et vérification

In [15]:
# Attendre quelques micro-batches pour voir les données
import time

print("Pipeline Silver en cours d'exécution...")
print("Attente de 30 secondes pour traiter les données...")

time.sleep(30)

# Vérifier le statut
print(f"\n✓ Statut de la query : {query_silver.status}")
print(f"✓ Dernière progression : {query_silver.lastProgress}")

25/12/17 15:12:54 ERROR MicroBatchExecution: Query [id = 96723a4a-a84b-41eb-a453-3d19f89e7603, runId = bbd11fac-c04c-4e47-8d79-398b6426f9fa] terminated with error
java.lang.NoClassDefFoundError: org/apache/spark/kafka010/KafkaConfigUpdater
	at org.apache.spark.sql.kafka010.KafkaSourceProvider$.kafkaParamsForDriver(KafkaSourceProvider.scala:645)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider$KafkaScan.toMicroBatchStream(KafkaSourceProvider.scala:482)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1.$anonfun$applyOrElse$4(MicroBatchExecution.scala:107)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1.applyOrElse(MicroBatchExecution.scala:100)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1.applyOrElse(MicroBatchExecution.scala:84)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$transformDownWithPruning$1(TreeNode.scala:5

Pipeline Silver en cours d'exécution...
Attente de 30 secondes pour traiter les données...

✓ Statut de la query : {'message': 'Terminated with exception: org/apache/spark/kafka010/KafkaConfigUpdater', 'isDataAvailable': False, 'isTriggerActive': False}
✓ Dernière progression : None


In [16]:
# Vérifier les données écrites dans Delta Lake Silver
from delta.tables import DeltaTable

print("📊 Vérification des données dans Delta Lake Silver...")

try:
    silver_df = spark.read.format("delta").load(DELTA_SILVER_PATH)
    
    total_count = silver_df.count()
    print(f"\n✓ Nombre total d'enregistrements dans Silver : {total_count}")
    
    if total_count > 0:
        print(f"\n✓ Aperçu des données :")
        silver_df.show(10, truncate=False)
        
        print(f"\n✓ Statistiques par building :")
        silver_df.groupBy("building_id").count().show()
        
        print(f"\n✓ Statistiques par type de capteur :")
        silver_df.groupBy("sensor_type").count().show()
        
        print(f"\n✓ Statistiques de qualité des données :")
        silver_df.groupBy("data_quality_score").count().show()
    else:
        print("⚠️  Aucune donnée trouvée dans Silver")
        print("💡 Vérifiez que :")
        print("   1. Le producer Kafka envoie des messages")
        print("   2. Les données passent le filtrage")
        print("   3. La pipeline fonctionne correctement")
        
except Exception as e:
    print(f"❌ Erreur lors de la lecture : {e}")

📊 Vérification des données dans Delta Lake Silver...
❌ Erreur lors de la lecture : Delta table `/opt/spark/delta/silver` doesn't exist.


## 11. Test de tolérance aux pannes

In [17]:
# Arrêter la query pour simuler une panne
# NOTE : Cette cellule arrête uniquement la pipeline Spark, pas le producteur Kafka
# Le producteur continue à produire des données (utile pour tester la reprise)
# Pour arrêter aussi le producteur, utilisez la cellule précédente "Arrêt du producteur Kafka"

if 'query_silver' in globals():
    query_silver.stop()
    print("✓ Pipeline arrêtée (simulation de panne)")
    print("💡 Le producteur Kafka continue à produire des données")
    print("💡 Pour l'arrêter aussi, utilisez la cellule 'Arrêt du producteur Kafka'")
else:
    print("⚠️  La query 'query_silver' n'est pas définie")
    print("💡 Lancez d'abord la pipeline avec la cellule 'Configuration de l'écriture vers Delta Lake Silver'")

# Relancer la pipeline - elle devrait reprendre depuis le checkpoint
print("\n✓ Redémarrage de la pipeline depuis le checkpoint...")

query_silver_restart = silver_stream \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH) \
    .option("path", DELTA_SILVER_PATH) \
    .partitionBy("building_id", "sensor_type") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✓ Pipeline redémarrée")
print("💡 Les offsets Kafka sont repris depuis le checkpoint")
print("💡 Aucune duplication de données grâce à la gestion des offsets")

✓ Pipeline arrêtée (simulation de panne)
💡 Le producteur Kafka continue à produire des données
💡 Pour l'arrêter aussi, utilisez la cellule 'Arrêt du producteur Kafka'

✓ Redémarrage de la pipeline depuis le checkpoint...
✓ Pipeline redémarrée
💡 Les offsets Kafka sont repris depuis le checkpoint
💡 Aucune duplication de données grâce à la gestion des offsets


25/12/17 15:13:28 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/12/17 15:13:28 ERROR MicroBatchExecution: Query [id = 96723a4a-a84b-41eb-a453-3d19f89e7603, runId = 5132525e-d5ae-435b-8b3f-6dd96af913e3] terminated with error
java.lang.NoClassDefFoundError: org/apache/spark/kafka010/KafkaConfigUpdater
	at org.apache.spark.sql.kafka010.KafkaSourceProvider$.kafkaParamsForDriver(KafkaSourceProvider.scala:645)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider$KafkaScan.toMicroBatchStream(KafkaSourceProvider.scala:482)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1.$anonfun$applyOrElse$4(MicroBatchExecution.scala:107)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1.applyOrElse(MicroBatchExecution.scala:100)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1

## 12. Documentation des concepts Kafka

### Offsets
Les **offsets** sont des identifiants uniques pour chaque message dans une partition Kafka. Ils permettent à Spark de :
- Suivre la position de lecture dans chaque partition
- Reprendre après une panne sans perdre de données
- Éviter la duplication grâce aux checkpoints

**Dans cette pipeline** : Les offsets sont gérés automatiquement via les checkpoints Spark. Si la pipeline s'arrête, elle reprend exactement où elle s'était arrêtée.

### Partitions
Les **partitions** permettent de :
- Distribuer les données sur plusieurs brokers Kafka
- Paralléliser le traitement (chaque partition peut être traitée indépendamment)
- Améliorer les performances et la scalabilité

**Dans cette pipeline** : Spark traite chaque partition en parallèle, améliorant le débit de traitement.

### Consumer Groups
Les **consumer groups** permettent de :
- Coordonner plusieurs consommateurs pour consommer un topic
- Répartir les partitions entre les consommateurs
- Gérer le rééquilibrage automatique en cas d'ajout/suppression de consommateurs

**Dans cette pipeline** : Spark utilise le consumer group `spark-streaming-consumer` pour consommer le topic. Les offsets sont suivis par consumer group.

### Intérêt d'un message broker dans une architecture temps réel
Un message broker comme Kafka apporte plusieurs avantages :
- **Découplage** : Les producteurs et consommateurs sont indépendants
- **Buffering** : Les messages sont stockés même si les consommateurs sont lents
- **Scalabilité** : Facile d'ajouter des producteurs/consommateurs
- **Tolérance aux pannes** : Les messages sont persistés et peuvent être rejoués
- **Débit élevé** : Capable de traiter des millions de messages par seconde